In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
spark=SparkSession.builder.appName("E2E_Pipeline").getOrCreate()

In [0]:
data = [
    (1, "C001", "Laptop", "50000", "2024-01-01"),
    (2, "C002", "Mobile", None, "2024-01-02"),
    (3, "C003", "Tablet", "20000", "2024-01-03"),
    (4, "C004", "Laptop", "55000", "2024-01-04"),
    (5, "C005", "Headphones", None, "2024-01-05"),
    (6, "C006", "Camera", "30000", "2024-01-06"),
    (7, "C007", "Mobile", "18000", "2024-01-07"),
    (8, "C008", "Watch", "8000", "2024-01-07")
    ]
columns=["Order_id","customer_id'","product","amount","updatred_date"]
df=spark.createDataFrame(data,columns)

Handling Nulls

In [0]:
df = df.fillna({"amount": 1000})
display(df)

Order_id,customer_id',product,amount,updatred_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,null,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,null,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


Task 2: Cast Columns
### Convert amount → integer
### Convert updated_date → date

In [0]:
df=df.withColumn("amount",col('amount').cast('int'))

In [0]:
df=df.withColumnRenamed("updated_date","date")
df.display()

Order_id,customer_id',product,amount,updatred_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,1000,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,1000,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


#Task 3: Add Derived Columns
### bonus = amount * 0.1
### category:
### 20000 → High
### else → Low


In [0]:
from pyspark.sql.functions import when

df = (df.withColumn("bonus", col('amount') * 0.1)
         .withColumn("Category",
                     when(col('amount') > 20000, "High")
                     .otherwise("Low")))
display(df)

Order_id,customer_id',product,amount,updatred_date,bonus,Category
1,C001,Laptop,50000,2024-01-01,5000.0,High
2,C002,Mobile,null,2024-01-02,null,Low
3,C003,Tablet,20000,2024-01-03,2000.0,Low
4,C004,Laptop,55000,2024-01-04,5500.0,High
5,C005,Headphones,null,2024-01-05,null,Low
6,C006,Camera,30000,2024-01-06,3000.0,High
7,C007,Mobile,18000,2024-01-07,1800.0,Low
8,C008,Watch,8000,2024-01-07,800.0,Low


Task 4: Apply UDF
### Create amount_bucket:
### < 10000 → Low
### 10000–30000 → Medium
### 30000 → High

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def amount_bucket(amount):
    if amount is None:
        return "Unknown"
    if amount < 10000:
        return "Low"
    elif 10000 <= amount <= 30000:
        return "Medium"
    else:
        return "High"

bucket_udf = udf(amount_bucket, StringType())
df = df.withColumn("amount_bucket", bucket_udf(col('amount')))
display(df)

Order_id,customer_id',product,amount,updatred_date,bonus,Category,amount_bucket
1,C001,Laptop,50000,2024-01-01,5000.0,High,High
2,C002,Mobile,null,2024-01-02,null,Low,Unknown
3,C003,Tablet,20000,2024-01-03,2000.0,Low,Medium
4,C004,Laptop,55000,2024-01-04,5500.0,High,High
5,C005,Headphones,null,2024-01-05,null,Low,Unknown
6,C006,Camera,30000,2024-01-06,3000.0,High,Medium
7,C007,Mobile,18000,2024-01-07,1800.0,Low,Medium
8,C008,Watch,8000,2024-01-07,800.0,Low,Low


Task-5
## Load all data to Target

In [0]:
# Save the transformed data to Delta table
target_table = "orders_processed"

df.write 
    .format("delta") 
    .mode("overwrite") 
    .saveAsTable(target_table)

print(f"Data successfully loaded to target table: {target_table}")
print(f"Total records written: {df.count()}")

Data successfully loaded to target table: orders_processed
Total records written: 8


Task-6:
## Incremental Load, load only updated records, handle duplicates

In [0]:
from delta.tables import DeltaTable

# Simulate new/updated data for incremental load
new_data = [
    (2, "C002", "Mobile", 15000, "2024-01-02"),  # Updated amount
    (9, "C009", "Speaker", 12000, "2024-01-08"),  # New record
    (10, "C010", "Keyboard", 5000, "2024-01-09")  # New record
]

columns = ["Order_id", "customer_id'", "product", "amount", "updatred_date"]
new_df = spark.createDataFrame(new_data, columns)

# Apply same transformations as original pipeline
new_df = new_df.fillna({"amount": 1000})
new_df = new_df.withColumn("amount", col('amount').cast('int'))

from pyspark.sql.functions import when, udf
from pyspark.sql.types import StringType

new_df = (new_df.withColumn("bonus", col('amount') * 0.1)
                .withColumn("Category",
                           when(col('amount') > 20000, "High")
                           .otherwise("Low")))

def amount_bucket(amount):
    if amount is None:
        return "Unknown"
    if amount < 10000:
        return "Low"
    elif 10000 <= amount <= 30000:
        return "Medium"
    else:
        return "High"

bucket_udf = udf(amount_bucket, StringType())
new_df = new_df.withColumn("amount_bucket", bucket_udf(col('amount')))

# Perform incremental load using MERGE (UPSERT)
target_table = "orders_processed"

# Load the existing Delta table
delta_table = DeltaTable.forName(spark, target_table)

# Merge new data with existing data
# Update if Order_id exists, Insert if it doesn't
delta_table.alias("target").merge(
    new_df.alias("source"),
    "target.Order_id = source.Order_id"  # Match condition on primary key
).whenMatchedUpdateAll(  # Update all columns when matched
).whenNotMatchedInsertAll(  # Insert new records when not matched
).execute()

print(f"Incremental load completed successfully!")
print(f"Records in new batch: {new_df.count()}")

# Verify the results
result_df = spark.table(target_table)
print(f"Total records in target table: {result_df.count()}")
display(result_df.orderBy("Order_id"))

Incremental load completed successfully!
Records in new batch: 3
Total records in target table: 10


Order_id,customer_id',product,amount,updatred_date,bonus,Category,amount_bucket
1,C001,Laptop,50000,2024-01-01,5000.0,High,High
2,C002,Mobile,15000,2024-01-02,1500.0,Low,Medium
3,C003,Tablet,20000,2024-01-03,2000.0,Low,Medium
4,C004,Laptop,55000,2024-01-04,5500.0,High,High
5,C005,Headphones,null,2024-01-05,null,Low,Unknown
6,C006,Camera,30000,2024-01-06,3000.0,High,Medium
7,C007,Mobile,18000,2024-01-07,1800.0,Low,Medium
8,C008,Watch,8000,2024-01-07,800.0,Low,Low
9,C009,Speaker,12000,2024-01-08,1200.0,Low,Medium
10,C010,Keyboard,5000,2024-01-09,500.0,Low,Low
